# S5.2 — RAG Chain

Interactive verification of the RAG chain service:
- RAG response models (SourceReference, RAGResponse)
- Citation-enforcing prompt templates
- RAGChain.aquery() with mocked dependencies
- RAGChain.aquery_stream() with mocked dependencies
- Factory function

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. RAG Response Models

In [2]:
from src.services.rag.models import SourceReference, RAGResponse, RetrievalMetadata, LLMMetadata

source = SourceReference(
    index=1,
    arxiv_id="1706.03762",
    title="Attention Is All You Need",
    authors=["Vaswani", "Shazeer", "Parmar"],
    arxiv_url="https://arxiv.org/abs/1706.03762",
    chunk_text="The dominant sequence transduction models are based on complex recurrent...",
    score=0.95,
)

response = RAGResponse(
    answer="Transformers use self-attention [1].",
    sources=[source],
    query="What are transformers?",
    retrieval_metadata=RetrievalMetadata(
        stages_executed=["hybrid_search", "rerank"],
        total_candidates=20,
        timings={"hybrid_search": 0.15, "rerank": 0.08},
    ),
    llm_metadata=LLMMetadata(provider="gemini", model="gemini-2.0-flash"),
)

assert response.answer == "Transformers use self-attention [1]."
assert response.sources[0].arxiv_url == "https://arxiv.org/abs/1706.03762"
assert response.retrieval_metadata.total_candidates == 20
print("\n\u2713 RAG response models work correctly")
print(f"  Answer: {response.answer}")
print(f"  Sources: {len(response.sources)}")
print(f"  Stages: {response.retrieval_metadata.stages_executed}")


✓ RAG response models work correctly
  Answer: Transformers use self-attention [1].
  Sources: 1
  Stages: ['hybrid_search', 'rerank']


## 2. Citation-Enforcing Prompt Templates

In [3]:
from src.services.rag.prompts import SYSTEM_PROMPT, build_user_prompt
from src.schemas.api.search import SearchHit

# Verify system prompt has citation instructions
assert "cite" in SYSTEM_PROMPT.lower() or "[1]" in SYSTEM_PROMPT
assert "arxiv" in SYSTEM_PROMPT.lower()
print("System Prompt (first 200 chars):")
print(SYSTEM_PROMPT[:200])

# Build user prompt with test hits
hits = [
    SearchHit(
        arxiv_id="1706.03762",
        title="Attention Is All You Need",
        authors=["Vaswani et al."],
        chunk_text="The Transformer is the first model to rely entirely on self-attention.",
    ),
    SearchHit(
        arxiv_id="1810.04805",
        title="BERT: Pre-training of Deep Bidirectional Transformers",
        authors=["Devlin et al."],
        chunk_text="BERT is designed to pre-train deep bidirectional representations.",
    ),
]

prompt = build_user_prompt(query="What are transformers?", search_hits=hits)
assert "1706.03762" in prompt
assert "Attention Is All You Need" in prompt
assert "self-attention" in prompt
print("\n\u2713 Prompt templates enforce citations and include context")
print(f"  Prompt length: {len(prompt)} chars")

System Prompt (first 200 chars):
You are a research assistant that answers questions based ONLY on the provided paper excerpts.

Rules:
1. Cite sources using inline [N] notation (e.g., [1], [2]) for every claim.
2. Only cite papers f

✓ Prompt templates enforce citations and include context
  Prompt length: 667 chars


## 3. RAGChain.aquery() with Mocked Dependencies

In [4]:
import asyncio
from unittest.mock import AsyncMock
from src.services.rag.chain import RAGChain
from src.services.llm.provider import LLMResponse, UsageMetadata
from src.services.retrieval.pipeline import RetrievalResult

# Mock LLM
mock_llm = AsyncMock()
mock_llm.generate = AsyncMock(return_value=LLMResponse(
    text="Transformers use self-attention [1] and have been extended to NLP [2].\n\n"
         "**Sources:**\n"
         "1. [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — Vaswani et al.\n"
         "2. [BERT](https://arxiv.org/abs/1810.04805) — Devlin et al.",
    model="gemini-2.0-flash",
    provider="gemini",
    usage=UsageMetadata(prompt_tokens=300, completion_tokens=100, total_tokens=400),
))

# Mock retrieval
mock_pipeline = AsyncMock()
mock_pipeline.retrieve = AsyncMock(return_value=RetrievalResult(
    results=hits,
    query="What are transformers?",
    stages_executed=["hybrid_search", "rerank"],
    total_candidates=15,
))

chain = RAGChain(llm_provider=mock_llm, retrieval_pipeline=mock_pipeline)
result = await chain.aquery("What are transformers?")

assert "[1]" in result.answer
assert len(result.sources) == 2
assert result.sources[0].arxiv_id == "1706.03762"
assert result.llm_metadata.provider == "gemini"
print("\n\u2713 RAGChain.aquery() returns correct RAGResponse")
print(f"  Answer preview: {result.answer[:80]}...")
print(f"  Sources: {[s.title for s in result.sources]}")
print(f"  LLM: {result.llm_metadata.provider}/{result.llm_metadata.model}")


✓ RAGChain.aquery() returns correct RAGResponse
  Answer preview: Transformers use self-attention [1] and have been extended to NLP [2].

**Source...
  Sources: ['Attention Is All You Need', 'BERT: Pre-training of Deep Bidirectional Transformers']
  LLM: gemini/gemini-2.0-flash


## 4. Empty Results (No Papers Found)

In [5]:
mock_pipeline_empty = AsyncMock()
mock_pipeline_empty.retrieve = AsyncMock(return_value=RetrievalResult(results=[], query="cats"))

chain_empty = RAGChain(llm_provider=mock_llm, retrieval_pipeline=mock_pipeline_empty)
result_empty = await chain_empty.aquery("cats")

assert "relevant papers" in result_empty.answer.lower()
assert len(result_empty.sources) == 0
print("\n\u2713 Empty results handled gracefully")
print(f"  Message: {result_empty.answer}")


✓ Empty results handled gracefully
  Message: I could not find any relevant papers in the knowledge base for your query. Try rephrasing your question or broadening your search terms.


## 5. Factory Function

In [6]:
from src.services.rag.factory import create_rag_chain

chain_from_factory = create_rag_chain(llm_provider=mock_llm, retrieval_pipeline=mock_pipeline)
assert isinstance(chain_from_factory, RAGChain)
print("\n\u2713 Factory function creates RAGChain correctly")


✓ Factory function creates RAGChain correctly


## 6. All Exports Available

In [7]:
from src.services.rag import (
    RAGChain,
    RAGResponse,
    SourceReference,
    RetrievalMetadata,
    LLMMetadata,
    create_rag_chain,
)

print("\n\u2713 All exports available from src.services.rag")
print(f"  Exports: RAGChain, RAGResponse, SourceReference, RetrievalMetadata, LLMMetadata, create_rag_chain")


✓ All exports available from src.services.rag
  Exports: RAGChain, RAGResponse, SourceReference, RetrievalMetadata, LLMMetadata, create_rag_chain
